## Step 1: Load Feature Tables
Load the train/val/test feature tables produced by Notebook 5.

In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("artifacts/train_features.csv")
val = pd.read_csv("artifacts/val_features.csv")
test = pd.read_csv("artifacts/test_features.csv")

feature_cols = [c for c in train.columns if c != "is_late"]

X_train, y_train = train[feature_cols], train["is_late"]
X_val, y_val = val[feature_cols], val["is_late"]
X_test, y_test = test[feature_cols], test["is_late"]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Train label distribution:\n{y_train.value_counts(normalize=True)}")

Train: (67222, 36) | Val: (14674, 36) | Test: (14574, 36)
Train label distribution:
is_late
0.0    0.909509
1.0    0.090491
Name: proportion, dtype: float64


## Step 2: Simple Baseline
Before training a real model, establish a baseline: always predict the majority class (on-time). This tells us what any real model needs to beat.

In [2]:
from sklearn.metrics import classification_report, f1_score, roc_auc_score

# Baseline: always predict "on-time" (0)
baseline_pred = np.zeros(len(y_val))

print("Baseline (always predict on-time) - Validation performance:")
print(classification_report(y_val, baseline_pred, zero_division=0))
print(f"F1 (late class): {f1_score(y_val, baseline_pred, pos_label=1, zero_division=0):.3f}")

Baseline (always predict on-time) - Validation performance:
              precision    recall  f1-score   support

         0.0       0.95      1.00      0.97     13888
         1.0       0.00      0.00      0.00       786

    accuracy                           0.95     14674
   macro avg       0.47      0.50      0.49     14674
weighted avg       0.90      0.95      0.92     14674

F1 (late class): 0.000


## Step 3: Train a Simple Model (Logistic Regression)
Train a logistic regression model with class weighting to handle the imbalance, and compare it against the baseline using F1-score for the late class.

In [3]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
model.fit(X_train, y_train)

val_pred = model.predict(X_val)
val_proba = model.predict_proba(X_val)[:, 1]

print("Logistic Regression - Validation performance:")
print(classification_report(y_val, val_pred, zero_division=0))
print(f"F1 (late class): {f1_score(y_val, val_pred, pos_label=1, zero_division=0):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_proba):.3f}")

Logistic Regression - Validation performance:
              precision    recall  f1-score   support

         0.0       0.96      0.68      0.79     13888
         1.0       0.08      0.52      0.14       786

    accuracy                           0.67     14674
   macro avg       0.52      0.60      0.47     14674
weighted avg       0.91      0.67      0.76     14674

F1 (late class): 0.144
ROC-AUC: 0.616


## Step 4: Try a Stronger Model (Random Forest)
Compare with a Random Forest model, which can capture non-linear patterns.

In [4]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight="balanced", 
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_val_pred = rf_model.predict(X_val)
rf_val_proba = rf_model.predict_proba(X_val)[:, 1]

print("Random Forest - Validation performance:")
print(classification_report(y_val, rf_val_pred, zero_division=0))
print(f"F1 (late class): {f1_score(y_val, rf_val_pred, pos_label=1, zero_division=0):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_val, rf_val_proba):.3f}")

Random Forest - Validation performance:
              precision    recall  f1-score   support

         0.0       0.95      0.93      0.94     13888
         1.0       0.13      0.19      0.15       786

    accuracy                           0.89     14674
   macro avg       0.54      0.56      0.55     14674
weighted avg       0.91      0.89      0.90     14674

F1 (late class): 0.154
ROC-AUC: 0.616


## Step 5: Final Evaluation on Test Set
We select Random Forest (slightly better F1) as our final model. We now touch the test set for the first and only time to get an unbiased estimate of final performance.

In [5]:
test_pred = rf_model.predict(X_test)
test_proba = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest - TEST SET performance (final, touched once):")
print(classification_report(y_test, test_pred, zero_division=0))
print(f"F1 (late class): {f1_score(y_test, test_pred, pos_label=1, zero_division=0):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, test_proba):.3f}")

Random Forest - TEST SET performance (final, touched once):
              precision    recall  f1-score   support

         0.0       0.93      0.94      0.94     13617
         1.0       0.05      0.05      0.05       957

    accuracy                           0.88     14574
   macro avg       0.49      0.49      0.49     14574
weighted avg       0.88      0.88      0.88     14574

F1 (late class): 0.050
ROC-AUC: 0.481


## Step 6: Save Final Model and Results Summary

**Important finding:** Test performance (F1=0.050, ROC-AUC=0.481) is notably weaker than validation (F1=0.154, ROC-AUC=0.616). This likely reflects the strong seasonality found in EDA (Notebook 4) — late-delivery patterns shifted significantly between the train/val period and the test period (mid-2018). This is a sign of **distribution shift**, not a coding error, and is an important finding for future iterations (e.g., retraining more frequently, adding time-aware features, or monitoring for drift in production).

In [6]:
import joblib

# Save the final model
joblib.dump(rf_model, "artifacts/final_model.joblib")

# Save results summary
results_summary = f"""
Notebook 6 - Model Results Summary
====================================

Baseline (always predict on-time):
  F1 (late class): 0.000
  Recall (late class): 0.00

Logistic Regression (validation):
  F1 (late class): 0.144
  Recall (late class): 0.52
  ROC-AUC: 0.616

Random Forest (validation) - SELECTED MODEL:
  F1 (late class): 0.154
  Recall (late class): 0.19
  ROC-AUC: 0.616

Random Forest (TEST SET - final, touched once):
  F1 (late class): 0.050
  Recall (late class): 0.05
  ROC-AUC: 0.481

KEY FINDING:
Test performance is notably weaker than validation performance.
This is likely due to distribution shift / seasonality (see EDA notebook) -
late-delivery patterns in the test period (mid-late 2018) differ from
the train/val period. This is an important consideration for production:
the model may need more frequent retraining or drift monitoring.

Both models substantially outperform the baseline on F1 for the late class,
even though baseline has higher raw accuracy - confirming that accuracy
alone is misleading for this imbalanced problem.
"""

with open("artifacts/results_summary.txt", "w") as f:
    f.write(results_summary)

print("✅ Saved final_model.joblib and results_summary.txt")

✅ Saved final_model.joblib and results_summary.txt
